## Leetcode 176. Second Highest Salary


Employee

| id | salary |
| ---|--------|
| 1  | 100    |
| 2  | 100    |
| 3  | 200    |
| 4  | 300    |

In [0]:
data = [(1, 100), (4, 100), (3, 200), (4, 300)]
Employee = spark.createDataFrame(data, ['id','salary'])
Employee.createOrReplaceTempView('Employee') # create a temp view


### Spark SQL

In [0]:
spark.sql('''
  SELECT MAX(salary) AS SecondHighestSalary
  FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)
''').show()


### Spark DataFrame

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
window = Window.orderBy(col('salary').desc())

Employee.withColumn('rank', dense_rank().over(window))\
    .filter(col('rank') == 2)\
        .select(col('salary').alias('SecondHighestSalary')).show()

### In Production

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "1") # decide partitions based on data size, or leave it to spark AQE

window = Window.orderBy(col('salary').desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)

result = Employee.withColumn('rank', dense_rank().over(window)) \
                 .filter(col('rank') == 2) \
                 .select(col('salary').alias('SecondHighestSalary')) \
                 .distinct() \
                 .limit(1) # handles duplicate rank 2's

if result.count() == 0:
    spark.createDataFrame([(None,)], ['SecondHighestSalary']).show()
else:
    result.show()